# Assignment 2 — Regression and Classification: Error Analysis

**Holon Institute of Technology (HIT) — Faculty of Computer Science**

| | |
|---|---|
| **Course** | Introduction to Data Science |
| **Assignment** | 2 — Regression and Classification: Error Analysis |
| **Student name** | Sean Mazon |
| **ID number** | 212234660 |
| **Lecturer** | Dr. Uri Itai |
| **Teaching assistant** | Hanit Ohayon Hadad |
| **Submission deadline** | 10.09.2026 |

---

## Table of Contents

0. [Introduction](#0-introduction)
1. [Data Preparation](#1-data-preparation)
2. [Regression Error Analysis](#2-regression-error-analysis)
3. [Regression Models](#3-regression-models)
4. [Classification Error Analysis](#4-classification-error-analysis)
5. [Final Reflection](#5-final-reflection)

## 0. Introduction

This chapter sets out what the assignment asks for, what data the analysis is
built on, and which parts of the setup are prescribed by the assignment as
opposed to chosen here. It contains no code and no results; the analysis begins
in chapter 1.

### 0.1 Objective

The assignment defines its own purpose as developing an understanding of
regression and classification models **through systematic error analysis**, and
asks students to distinguish between failures that arise from data issues, from
model assumptions, and from how the problem was formulated.

That framing determines how this notebook is written. **Predictive accuracy is
not the goal.** A model that scored well but whose errors could not be
explained would fail the assignment, while a model that performs modestly and
whose failures are understood would satisfy it. Every metric reported here
exists to support a statement about *why* the model is wrong where it is wrong.

The assignment prescribes four things, and they are the fixed points of this
notebook:

1. The four analytical sections and their sub-tasks — residual analysis, error
   as a function of features, extreme errors, statistical properties of errors,
   the regression model comparison, the classification error analysis, and the
   final reflection.
2. Three families of regression model: linear, decision tree, and at least one
   ensemble.
3. That **all analyses be conducted using k-fold cross-validation**.
4. That the choice of *k* be justified in terms of dataset size, computational
   complexity, and the bias–variance trade-off.

Everything else — which dataset, which target variables, which features, which
classifiers — is not specified by the assignment. Those are decisions taken
here, and each one is identified as such and justified where it is made.

### 0.2 Dataset Overview

The assignment does not supply or name a dataset. This analysis continues with
the dataset used in Assignment 1: **1,500 of the highest-revenue games released
on Steam during 2024**, obtained from Kaggle and extracted on 2024-09-09. The
file holds 1,500 rows and 11 columns, mixing numeric, temporal and categorical
variables.

Using the same data is a deliberate choice. Assignment 1 established a set of
findings about its structure that directly constrain any model built on it, and
carrying those forward is more informative than rediscovering them:

| Finding from Assignment 1 | Consequence for this assignment |
|---|---|
| `reviewScore = 0` is a placeholder in 99 rows, `avgPlaytime = 0` in one | Must become missing values, then be imputed **inside** the cross-validation folds |
| `revenue` is extremely heavy-tailed (skew 22.9; 1.19 after a log transform) | Motivates the regression target chosen in 0.3 |
| `publishers` and `developers` hold 1,169 and 1,517 distinct companies | Cannot be one-hot encoded; must be aggregated into numeric features |
| `publisherClass` has a single `Hobbyist` row | Merged into `Indie`, leaving three ordered levels |
| Three missing cells, all in the company fields | Imputation strategy needed, though the volume is negligible |
| The rows were selected *because* revenue was high | The most important caveat in this notebook — see below |

**The survivorship caveat, stated at the outset.** These 1,500 games were
selected on revenue, which is the same quantity this notebook sets out to
predict and to classify. Every result that follows therefore describes the
behaviour of models *within an already-successful population*, never across
Steam as a whole. Statements such as "the model predicts commercial success"
must be read as "the model separates the strongest performers from the
moderately strong ones". This is not a limitation discovered at the end of the
analysis; it is a property of the sample and it is carried through every
chapter.

The full structural analysis of this dataset is in the Assignment 1 notebook,
`assignment1-eda/notebooks/steam_2024_eda.ipynb`.

### 0.3 Problem Definitions

**The assignment specifies neither a regression target nor a classification
target.** Both are defined here, and both are methodological choices rather than
requirements. Each is justified in full in chapter 1, where the supporting
evidence is computed; the definitions are stated here so the rest of the
notebook can refer to them.

#### Regression problem

> Predict **`log10(revenue)`** — the base-10 logarithm of a game's estimated
> revenue in US dollars.

The logarithm is used because Assignment 1 measured a skew of 22.9 on the raw
values against 1.19 after the transform. Residual analysis on the raw scale
would be dominated by a handful of titles and would make the questions in
section 2.1 — about centring, systematic pattern and heteroscedasticity —
effectively unanswerable. Metrics are also reported back on the original dollar
scale so that the error magnitudes remain interpretable.

#### Classification problem

> Predict whether a game is a **commercial standout**, defined as
> `revenue` at or above the 75th percentile of the dataset.

This yields roughly 375 positive and 1,125 negative cases. A threshold at the
median would produce a balanced problem, but a 25/75 split is the more
informative choice for what section 3 of the assignment asks: it gives the
precision–recall trade-off something to trade, and it makes the threshold sweep
and the Matthews correlation coefficient meaningful rather than near-symmetric.

The two error types also carry genuinely different costs in the underlying
decision problem, which section 4.2 develops. A false positive corresponds to
backing a title that does not deliver; a false negative corresponds to passing
on one that would have. The threshold is a fixed definitional choice computed
once on the full dataset, not a quantity learned from the data.

#### A note on the feature set

The features are **not fixed at this point**. Assignment 1 found that `revenue`,
`copiesSold` and `price` are tightly linked, which raises the possibility that
including `copiesSold` as a predictor would make both problems trivial and leave
no error structure to analyse. Section 1 tests that empirically and decides the
feature set on the evidence, rather than assuming the answer here.

### 0.4 Methodology

Each analytical section follows the same four-part pattern:

- **Method** — what is being done and why.
- **Results** — the metrics, tables and figures.
- **Interpretation** — what the results mean, including where they contradict
  what was expected.
- **Conclusion** — the single insight the section establishes.

Two rules govern the writing. First, **interpretations are written after the
results are read**, never drafted in advance; where an outcome is surprising it
is reported as surprising rather than smoothed over. Second, **evidence and
hypothesis are kept distinct**: where the data shows a pattern but cannot
establish its cause, the proposed cause is labelled as a hypothesis and the
information that would test it is named.

Chapter 3 of the assignment requires three families of regression model. This
notebook uses Linear Regression, a Decision Tree Regressor, and a **Random
Forest Regressor** as the ensemble. All three are trained on an identical
feature set under an identical resampling protocol, so that differences in
performance are attributable to the models rather than to their inputs.

For the classification analysis the assignment names no model. Two are used
here — **Logistic Regression** and a **Random Forest Classifier** — because the
discussion section calls for a comparison, and a comparison needs more than one
subject. This is a choice, not a requirement.

### 0.5 Cross-Validation Strategy

The assignment requires that **all** analyses use k-fold cross-validation. That
requirement is stated in its objective rather than in the modelling section, so
it governs the error analysis as much as the model comparison: the residuals
examined in chapter 2 and the confusion matrix in chapter 4 must not be computed
from predictions the model made on data it was trained on.

The protocol used throughout is therefore:

- **Out-of-fold prediction.** Every observation receives a prediction from a
  model that never saw it during training. Pooling these across folds yields one
  prediction per row, which is what the residual analysis, the extreme-error
  analysis and the confusion matrix are built from.
- **k = 10**, with shuffling and a fixed random seed. The justification in terms
  of dataset size, computational cost and the bias–variance trade-off is given
  in section 1, where it can be supported with the actual dimensions of the data
  rather than asserted here.
- **Stratified folds for classification**, so that the 25/75 class balance is
  preserved in every fold and the per-fold metrics remain comparable.
- **Pipelines for all preprocessing.** Imputation, scaling and encoding are
  fitted on the training portion of each fold only. Fitting them on the full
  dataset first — the most common source of leakage in cross-validated work —
  would let information from the held-out fold influence the transformation
  applied to it.

Section 1 sets this protocol up and states explicitly what each element
prevents.

## 1. Data Preparation

**Method.** This chapter assembles the data the rest of the notebook depends on
and settles three questions before any model is compared: which features the
models may see, how many folds to use, and how preprocessing is prevented from
leaking information between them. Each is decided from measurements on this
dataset rather than by convention.

### 1.1 Loading and Cleaning

The cleaning established in Assignment 1 is imported rather than rewritten, so
that both notebooks treat the data identically. It converts the placeholder
zeros in `reviewScore` and `avgPlaytime` into missing values, parses the release
date, orders `publisherClass`, and merges the single `Hobbyist` row into `Indie`.

Critically, the placeholders become **missing values rather than imputed
values** at this stage. Imputation belongs inside the cross-validation folds,
for the reason set out in section 1.5.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, StratifiedKFold

# The helper package for this assignment sits one level above the notebook. It
# in turn adds Assignment 1's package to the path, so the cleaning defined there
# can be reused unchanged.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from error_analysis.data_setup import (
    ALL_FEATURES,
    BOOLEAN_FEATURES,
    CATEGORICAL_FEATURES,
    CLASSIFICATION_TARGET,
    NUMERIC_FEATURES,
    REGRESSION_TARGET,
    STANDOUT_QUANTILE,
    WITHHELD_COLUMNS,
    build_modelling_frame,
    build_pipeline,
    describe_feature_roles,
)
from error_analysis.validation import (
    RANDOM_STATE,
    compare_feature_sets,
    compare_fold_counts,
    regression_metrics,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

games = build_modelling_frame()

print(f"Rows: {len(games):,}    Columns available after derivation: {games.shape[1]}")
print()
print("Missing values in the candidate feature matrix:")
print(games[ALL_FEATURES].isna().sum().loc[lambda counts: counts > 0].to_string())

Rows: 1,500    Columns available after derivation: 30

Missing values in the candidate feature matrix:
reviewScore    99
avgPlaytime     1


**Results and interpretation.** The frame holds all 1,500 games. Two feature
columns carry missing values, and both are the placeholders identified in
Assignment 1 rather than genuine gaps in the source file: 99 review scores that
were stored as `0`, and one average playtime. No rows are dropped — discarding
99 games because their review score is unknown would remove 6.6% of the data and,
as Assignment 1 showed, that group is not a random sample of the whole.

### 1.2 Feature and Target Definition

**Method.** The assignment specifies neither targets nor features. Both are
defined here, and the same feature set is used for every model and for both
problems, which is what makes the comparison in chapter 3 a fair one.

In [2]:
display(describe_feature_roles())

standout_threshold = games["revenue"].quantile(STANDOUT_QUANTILE)
class_counts = games[CLASSIFICATION_TARGET].value_counts().sort_index()

print(f"Regression target      : {REGRESSION_TARGET}  "
      f"(range {games[REGRESSION_TARGET].min():.2f} to {games[REGRESSION_TARGET].max():.2f}, "
      f"sd {games[REGRESSION_TARGET].std():.2f})")
print(f"Classification target  : {CLASSIFICATION_TARGET} = revenue >= ${standout_threshold:,.0f} "
      f"({STANDOUT_QUANTILE:.0%} percentile)")
print(f"Class balance          : {class_counts[1]} standouts / {class_counts[0]} others "
      f"({games[CLASSIFICATION_TARGET].mean():.1%} positive)")

,Feature,Type,Why it is included
0,price,numeric,Listed price; the strongest legitimate predict...
1,reviewScore,numeric,"Percentage of positive reviews, after placehol..."
2,avgPlaytime,numeric,"Average hours played, a proxy for engagement"
3,daysOnSale,numeric,Exposure at the snapshot date
4,publisherReleaseCount,numeric,How many titles the publisher has in this dataset
5,releaseMonth,numeric,Calendar month of release
6,plotClass,categorical,"Studio scale: Indie, AA or AAA"
7,releaseWeekday,categorical,Day of the week the title launched
8,selfPublished,boolean,Whether publisher and developer are the same c...
9,isFreeToPlay,boolean,Whether the listed price is zero


Regression target      : logRevenue  (range 4.32 to 8.92, sd 0.75)
Classification target  : isStandout = revenue >= $455,157 (75% percentile)
Class balance          : 375 standouts / 1125 others (25.0% positive)


**Interpretation.** Ten features are available to every model: six numeric, two
categorical and two boolean.

`log10(revenue)` spans 4.32 to 8.92 with a standard deviation of 0.75, which
means the target covers more than four orders of magnitude on the dollar scale.
The classification threshold falls at **$455,157**, giving 375 standouts against
1,125 others — the 25/75 split chosen in section 0.3.

**One feature carries a caveat that should be stated rather than buried.**
`publisherReleaseCount` counts how many titles a publisher has *in this dataset*,
and this dataset was selected on revenue. A studio therefore appears often partly
because it had several commercially successful releases, so the feature carries a
faint trace of the target. Its rank correlation with `log10(revenue)` is only
0.12, so the effect is weak, but it is not zero and it is worth remembering when
this feature appears in the error analysis.

### 1.3 Testing for Target Leakage

**Method.** Assignment 1 found `revenue`, `copiesSold` and `price` to be closely
linked, which raises a question that has to be settled before any model is built:
would including `copiesSold` as a predictor leave any error structure to analyse?

The question is answered in two steps rather than assumed. First, the strength of
the relationship is measured directly. Second, the identical Linear Regression is
run under the identical protocol twice, changing only whether `copiesSold` is
available to it.

In [3]:
correlation_rows = []
for column in ["copiesSold", "price", "avgPlaytime", "reviewScore", "daysOnSale",
               "publisherReleaseCount"]:
    subset = games[[column, REGRESSION_TARGET]].dropna()
    values = np.log10(subset[column]) if column == "copiesSold" else subset[column]
    correlation_rows.append({
        "Feature": f"log10({column})" if column == "copiesSold" else column,
        "Pearson vs log10(revenue)": round(pearsonr(values, subset[REGRESSION_TARGET])[0], 3),
        "Spearman": round(spearmanr(values, subset[REGRESSION_TARGET])[0], 3),
    })
display(pd.DataFrame(correlation_rows))

# If revenue were simply copies multiplied by price, regressing the logarithms
# against each other would recover coefficients of 1.0 and an R2 close to 1.
paid_games = games[games["price"] > 0]
identity_features = pd.DataFrame({
    "log10(copiesSold)": np.log10(paid_games["copiesSold"]),
    "log10(price)": np.log10(paid_games["price"]),
})
identity_model = LinearRegression().fit(identity_features, paid_games[REGRESSION_TARGET])

print(f"log10(revenue) ~ log10(copiesSold) + log10(price), paid games only (n={len(paid_games):,})")
print(f"  R2           : {identity_model.score(identity_features, paid_games[REGRESSION_TARGET]):.4f}")
print(f"  coefficients : log10(copiesSold) = {identity_model.coef_[0]:.3f}, "
      f"log10(price) = {identity_model.coef_[1]:.3f}")
print( "  an exact accounting identity would give 1.000 and 1.000")

,Feature,Pearson vs log10(revenue),Spearman
0,log10(copiesSold),0.897,0.869
1,price,0.398,0.312
2,avgPlaytime,0.275,0.440
3,reviewScore,0.034,0.037
4,daysOnSale,0.043,0.041
5,publisherReleaseCount,0.035,0.120


log10(revenue) ~ log10(copiesSold) + log10(price), paid games only (n=1,415)
  R2           : 0.9901
  coefficients : log10(copiesSold) = 0.988, log10(price) = 1.005
  an exact accounting identity would give 1.000 and 1.000


In [4]:
leakage_comparison = compare_feature_sets(
    games,
    games[REGRESSION_TARGET],
    {
        "copiesSold included": NUMERIC_FEATURES + ["logCopiesSold"],
        "copiesSold withheld": NUMERIC_FEATURES,
    },
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
)
leakage_comparison

,R2,MAE (log10),RMSE (log10),Median error factor,Residual SD,Excess kurtosis
Feature set,,,,,,
copiesSold included,0.9266,0.1229,0.2024,1.21,0.2024,27.71
copiesSold withheld,0.2855,0.4910,0.6313,2.49,0.6315,0.85


**Results.** The correlation table shows `copiesSold` at 0.897 against the
target, far ahead of the next strongest feature at 0.398. On its own that would
only mark it as a strong predictor. The regression that follows shows it is
something else.

Fitting `log10(revenue)` on `log10(copiesSold)` and `log10(price)` for the 1,415
paid games returns **R² = 0.9901 with coefficients of 0.988 and 1.005**. Those
coefficients are the finding. A coefficient of 1.0 on both logged terms is
exactly what an accounting identity of the form `revenue = copies × price`
produces, and the fitted values sit within about one percent of it.

The cross-validated comparison confirms what that implies for modelling:

| | R² | MAE (log₁₀) | Median error factor | Residual SD | Excess kurtosis |
|---|---|---|---|---|---|
| `copiesSold` included | 0.927 | 0.123 | ×1.21 | 0.202 | 27.7 |
| `copiesSold` withheld | 0.286 | 0.491 | ×2.49 | 0.631 | 0.8 |

**Interpretation.** Including `copiesSold` produces a model that appears
excellent and is analytically worthless. Its R² of 0.927 does not reflect
understanding of what makes a game succeed; it reflects the model rediscovering
an arithmetic relationship between three columns that were derived from one
another. The excess kurtosis of 27.7 is the giveaway: almost every residual is
near zero, with a small number of large exceptions. Those exceptions are the
free-to-play titles, where `price = 0` breaks the identity — as Assignment 1
established, 85 games earn revenue that no product of price and copies can
generate.

An error analysis built on that model would be an analysis of one accounting
rule and its 85 exceptions. Sections 2 and 4 would have almost nothing to
examine.

Withholding `copiesSold` gives a genuine predictive problem. R² falls to 0.286,
which is modest, and the typical prediction is wrong by a factor of about 2.5.
But the residuals are far better behaved — an excess kurtosis of 0.8 rather than
27.7 — which means the errors are distributed rather than concentrated in a few
identity-breaking rows, and there is real structure for chapter 2 to investigate.

**Conclusion.** `copiesSold` is withheld, along with every column computed from
revenue. This is a decision about *problem formulation*, not a technical
convenience: the question this notebook asks is whether a game's commercial
outcome can be predicted from its observable characteristics, and a predictor
that is measured at the same moment as the target cannot be part of that
question. The cost is a much lower R², and that cost is accepted deliberately.

In [5]:
print("Columns withheld from every model:")
for column in WITHHELD_COLUMNS:
    print(f"  {column}")

Columns withheld from every model:
  revenue
  logRevenue
  copiesSold
  logCopiesSold
  revenuePerCopy
  revenuePerDay
  priceRealisation
  revenueQuartile
  isStandout


### 1.4 Choice of k

**Method.** The assignment requires the choice of *k* to be justified in terms
of dataset size, computational complexity and the bias–variance trade-off.
Rather than assert a conventional value, the baseline model is run at four
values of *k* and the three quantities are measured directly.

In [6]:
fold_comparison = compare_fold_counts(games[ALL_FEATURES], games[REGRESSION_TARGET])
fold_comparison

,Training rows per fold,Test rows per fold,Mean R2,SD of R2 across folds,Runtime (s)
k,,,,,
3,1000,500,0.2839,0.0156,0.02
5,1200,300,0.2816,0.0519,0.03
10,1350,150,0.2807,0.0631,0.06
20,1425,75,0.2629,0.1144,0.13


**Results and interpretation.** The measurements do not support the usual
default of k = 10 for this dataset.

**Bias.** Mean R² is essentially flat from k = 3 to k = 10, moving only from
0.2839 to 0.2807 while training-set size grows from 1,000 rows to 1,350. The
model is already on the flat part of its learning curve, so the standard argument
for a larger *k* — that bigger training folds reduce pessimistic bias — buys
nothing here. At k = 20 the mean actually falls to 0.2629.

**Variance.** The spread of R² across folds rises steadily with *k*: 0.016 at
k = 3, 0.052 at k = 5, 0.063 at k = 10, and 0.114 at k = 20. This is the
expected consequence of smaller test folds. At k = 20 each fold scores only 75
games, and with a target spanning four orders of magnitude a handful of large
titles landing in one fold moves its R² substantially.

**Computational complexity.** Every configuration runs in well under a second,
so compute does not constrain the choice for these models. It is worth noting
that this stops being true in chapter 3, where a Random Forest is fitted once per
fold.

**Choice: k = 5.** It costs nothing in bias relative to k = 10 — the difference
in mean R² is 0.001 — while reducing the fold-to-fold spread by roughly a fifth.
It leaves 300 games in each test fold, and for the classification problem it
places **75 positive cases in every stratified fold** rather than the 37 that
k = 10 would give, which matters for the stability of precision and recall in
chapter 4. k = 3 has still lower variance but trains on only 1,000 rows and is
an unusual choice that would need defending on its own terms.

**Conclusion.** All subsequent analysis uses 5-fold cross-validation, stratified
for classification, with `shuffle=True` and a fixed seed so the folds are
reproducible.

In [7]:
REGRESSION_CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
CLASSIFICATION_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

positives_per_fold = [
    int(games[CLASSIFICATION_TARGET].iloc[test_index].sum())
    for _, test_index in CLASSIFICATION_CV.split(games[ALL_FEATURES], games[CLASSIFICATION_TARGET])
]
print(f"Positive cases in each stratified test fold: {positives_per_fold}")

Positive cases in each stratified test fold: [75, 75, 75, 75, 75]


### 1.5 Preventing Leakage Between Folds

**Method.** Section 1.3 dealt with leakage from the target into the features.
A second and more subtle form arises from preprocessing. If the median used to
impute `reviewScore` were computed once on all 1,500 games, every training fold
would contain a value derived partly from the games it is about to be tested on.
The measured performance would then be optimistic in a way no metric would
reveal.

Every transformation is therefore placed inside a pipeline, which scikit-learn
fits on the training portion of each fold and only then applies to the held-out
portion.

In [8]:
example_pipeline = build_pipeline(LinearRegression())
example_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('estimator', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

**Interpretation.** Each element of the pipeline prevents a specific failure:

| Step | Applies to | What it prevents |
|---|---|---|
| `SimpleImputer(strategy="median")` | 6 numeric columns | A median computed across all folds carrying information from the held-out data |
| `StandardScaler` | 6 numeric columns | A mean and standard deviation fitted on the full dataset doing the same |
| `OneHotEncoder(handle_unknown="ignore")` | `plotClass`, `releaseWeekday` | A category appearing only in the test fold raising an error rather than being handled |
| `passthrough` | 2 boolean columns | Unnecessary transformation of already-binary values |

The median is the right imputation strategy for these columns because both are
skewed — `avgPlaytime` had a skew of 7.2 in Assignment 1 — and a mean would be
pulled by the same long tail that motivated the log transform of the target.

Two further points complete the leakage argument. **Predictions come from
`cross_val_predict`**, so every residual analysed in chapter 2 and every entry in
the confusion matrix of chapter 4 was produced by a model that had not seen that
row during training. And the **classification threshold is a definitional choice,
not a learned parameter**: the 75th percentile is computed once on the full
dataset to define what "standout" means, in the same way a class label would be
fixed in advance. It is not re-estimated per fold, because a label that changed
meaning between folds could not be compared across them.

**Conclusion.** The data is prepared, the feature set is decided on evidence, and
the protocol is fixed: 5-fold cross-validation, stratified where the problem is
a classification, with all preprocessing confined to the training folds and all
error analysis conducted on out-of-fold predictions.

## 2. Regression Error Analysis

*Not yet written.* Residual analysis, error as a function of features, the top 5% of absolute errors, and the statistical properties of the error distribution.

## 3. Regression Models

*Not yet written.* Linear Regression, Decision Tree and Random Forest compared on a common feature set, followed by the critical discussion and the choice of a preferred model.

## 4. Classification Error Analysis

*Not yet written.* Confusion matrix, false positives and negatives, probability-based analysis, error as a function of features, threshold sensitivity, and ROC/AUC with the Matthews correlation coefficient.

## 5. Final Reflection

*Not yet written.* The four closing questions, answered explicitly.